In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import json
import os

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

COLORS = {
    'Doc': '#4C72B0', 'Img': '#DD8452', 'Movie': '#55A868',
    'Rec': '#C44E52', 'BGM': '#8172B3'
}
DPI = 300

# Load incremental data
with open('../MR_TriCHEF/results/incremental_movie.json', 'r') as f:
    movie_inc = json.load(f)

with open('../MR_TriCHEF/results/incremental_music.json', 'r') as f:
    music_inc = json.load(f)

In [ ]:
# fig07_incremental_movie.png - Movie domain incremental scaling

steps = movie_inc['steps']
n_files_list = [s['n_files'] for s in steps]

# Collect all unique query names
query_names = []
for s in steps:
    for q in s['queries']:
        if q['query'] not in query_names:
            query_names.append(q['query'])

# Build confidence matrix: shape (n_queries, n_steps)
conf_matrix = np.full((len(query_names), len(steps)), np.nan)
for si, s in enumerate(steps):
    for q in s['queries']:
        if q['query'] in query_names:
            qi = query_names.index(q['query'])
            conf_matrix[qi, si] = q['top1_conf']

# Calibration params
mu_null_list = [s['calibration']['mu_null'] for s in steps]
sigma_null_list = [s['calibration']['sigma_null'] for s in steps]

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.suptitle('Movie 도메인 증분 인덱싱에 따른 신뢰도 변화', fontsize=15, fontweight='bold', y=1.01)

# Top subplot: confidence per query
ax0 = axes[0]
cmap = plt.get_cmap('tab10')
for qi, qname in enumerate(query_names):
    label = (qname[:30] + '...') if len(qname) > 30 else qname
    color = cmap(qi % 10)
    vals = conf_matrix[qi, :]
    valid_mask = ~np.isnan(vals)
    ax0.plot(
        np.array(n_files_list)[valid_mask],
        vals[valid_mask],
        marker='o',
        color=color,
        label=label,
        linewidth=1.8,
        markersize=5
    )

ax0.set_xlabel('인덱싱 파일 수', fontsize=12)
ax0.set_ylabel('신뢰도 점수', fontsize=12)
ax0.set_title('쿼리별 신뢰도 변화', fontsize=13)
ax0.set_xticks(n_files_list)
ax0.legend(loc='upper left', fontsize=8, ncol=2, framealpha=0.7)
ax0.grid(True, alpha=0.3)

# Bottom subplot: calibration params (dual y-axis)
ax1 = axes[1]
color_mu = '#4C72B0'
color_sigma = '#DD8452'

line1, = ax1.plot(n_files_list, mu_null_list, marker='s', color=color_mu,
                  label='mu_null', linewidth=2, markersize=6)
ax1.set_xlabel('인덱싱 파일 수', fontsize=12)
ax1.set_ylabel('mu_null', fontsize=12, color=color_mu)
ax1.tick_params(axis='y', labelcolor=color_mu)
ax1.set_xticks(n_files_list)

ax1b = ax1.twinx()
line2, = ax1b.plot(n_files_list, sigma_null_list, marker='^', color=color_sigma,
                   label='sigma_null', linewidth=2, markersize=6)
ax1b.set_ylabel('sigma_null', fontsize=12, color=color_sigma)
ax1b.tick_params(axis='y', labelcolor=color_sigma)

ax1.set_title('캘리브레이션 파라미터 변화 (mu_null, sigma_null)', fontsize=13)
lines = [line1, line2]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left', fontsize=10)
ax1.grid(True, alpha=0.3)

plt.tight_layout()
out_path = 'fig07_incremental_movie.png'
plt.savefig(out_path, dpi=DPI, bbox_inches='tight')
plt.close()
print(f'Saved: {out_path}')

In [ ]:
# fig07_incremental_music.png - Music domain incremental scaling

steps_m = music_inc['steps']
n_files_list_m = [s['n_files'] for s in steps_m]

# Collect all unique query names
query_names_m = []
for s in steps_m:
    for q in s['queries']:
        if q['query'] not in query_names_m:
            query_names_m.append(q['query'])

# Build confidence matrix: shape (n_queries, n_steps)
conf_matrix_m = np.full((len(query_names_m), len(steps_m)), np.nan)
for si, s in enumerate(steps_m):
    for q in s['queries']:
        if q['query'] in query_names_m:
            qi = query_names_m.index(q['query'])
            conf_matrix_m[qi, si] = q['top1_conf']

# Calibration params
mu_null_list_m = [s['calibration']['mu_null'] for s in steps_m]
sigma_null_list_m = [s['calibration']['sigma_null'] for s in steps_m]

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.suptitle('Music 도메인 증분 인덱싱에 따른 신뢰도 변화', fontsize=15, fontweight='bold', y=1.01)

# Top subplot: confidence per query
ax0 = axes[0]
cmap = plt.get_cmap('tab10')
for qi, qname in enumerate(query_names_m):
    label = (qname[:30] + '...') if len(qname) > 30 else qname
    color = cmap(qi % 10)
    vals = conf_matrix_m[qi, :]
    valid_mask = ~np.isnan(vals)
    ax0.plot(
        np.array(n_files_list_m)[valid_mask],
        vals[valid_mask],
        marker='o',
        color=color,
        label=label,
        linewidth=1.8,
        markersize=5
    )

ax0.set_xlabel('인덱싱 파일 수', fontsize=12)
ax0.set_ylabel('신뢰도 점수', fontsize=12)
ax0.set_title('쿼리별 신뢰도 변화', fontsize=13)
ax0.set_xticks(n_files_list_m)
ax0.legend(loc='upper left', fontsize=8, ncol=2, framealpha=0.7)
ax0.grid(True, alpha=0.3)

# Bottom subplot: calibration params (dual y-axis)
ax1 = axes[1]
color_mu = '#4C72B0'
color_sigma = '#DD8452'

line1, = ax1.plot(n_files_list_m, mu_null_list_m, marker='s', color=color_mu,
                  label='mu_null', linewidth=2, markersize=6)
ax1.set_xlabel('인덱싱 파일 수', fontsize=12)
ax1.set_ylabel('mu_null', fontsize=12, color=color_mu)
ax1.tick_params(axis='y', labelcolor=color_mu)
ax1.set_xticks(n_files_list_m)

ax1b = ax1.twinx()
line2, = ax1b.plot(n_files_list_m, sigma_null_list_m, marker='^', color=color_sigma,
                   label='sigma_null', linewidth=2, markersize=6)
ax1b.set_ylabel('sigma_null', fontsize=12, color=color_sigma)
ax1b.tick_params(axis='y', labelcolor=color_sigma)

ax1.set_title('캘리브레이션 파라미터 변화 (mu_null, sigma_null)', fontsize=13)
lines = [line1, line2]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left', fontsize=10)
ax1.grid(True, alpha=0.3)

plt.tight_layout()
out_path = 'fig07_incremental_music.png'
plt.savefig(out_path, dpi=DPI, bbox_inches='tight')
plt.close()
print(f'Saved: {out_path}')

In [ ]:
# fig07_scaling_comparison.png - Movie vs Music comparison

# Compute mean confidence per step for Movie
steps_movie = movie_inc['steps']
n_files_movie = [s['n_files'] for s in steps_movie]
mean_conf_movie = []
min_conf_movie = []
max_conf_movie = []
for s in steps_movie:
    confs = [q['top1_conf'] for q in s['queries']]
    mean_conf_movie.append(np.mean(confs))
    min_conf_movie.append(np.min(confs))
    max_conf_movie.append(np.max(confs))

# Compute mean confidence per step for Music
steps_music = music_inc['steps']
n_files_music = [s['n_files'] for s in steps_music]
mean_conf_music = []
min_conf_music = []
max_conf_music = []
for s in steps_music:
    confs = [q['top1_conf'] for q in s['queries']]
    mean_conf_music.append(np.mean(confs))
    min_conf_music.append(np.min(confs))
    max_conf_music.append(np.max(confs))

fig, ax = plt.subplots(figsize=(12, 7))

# Movie line
ax.plot(n_files_movie, mean_conf_movie, marker='o', color=COLORS['Movie'],
        label='Movie', linewidth=2.5, markersize=7, zorder=3)
ax.fill_between(n_files_movie, min_conf_movie, max_conf_movie,
                color=COLORS['Movie'], alpha=0.15, label='Movie (min-max)')

# Music line
ax.plot(n_files_music, mean_conf_music, marker='s', color=COLORS['BGM'],
        label='Music', linewidth=2.5, markersize=7, zorder=3)
ax.fill_between(n_files_music, min_conf_music, max_conf_music,
                color=COLORS['BGM'], alpha=0.15, label='Music (min-max)')

ax.set_xlabel('인덱싱 파일 수', fontsize=13)
ax.set_ylabel('평균 신뢰도 점수', fontsize=13)
ax.set_title('Movie vs Music 증분 스케일링 비교', fontsize=15, fontweight='bold')
ax.legend(fontsize=11, framealpha=0.8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
out_path = 'fig07_scaling_comparison.png'
plt.savefig(out_path, dpi=DPI, bbox_inches='tight')
plt.close()
print(f'Saved: {out_path}')